# Installation / Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install huggingface_hub
!pip install tabulate datasets
!pip install cupy-cuda12x

In [ ]:
from huggingface_hub import login
login(token="HF_TOKEN")

In [ ]:
!git clone https://github.com/danielliu5670/company_similarity_sae

Cloning into 'company_similarity_sae'...
remote: Enumerating objects: 1415, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 1415 (delta 12), reused 15 (delta 9), pack-reused 1394 (from 1)
Receiving objects: 100% (1415/1415), 14.04 MiB | 24.16 MiB/s, done.
Resolving deltas: 100% (235/235), done.
Filtering content: 100% (38/38), 238.78 MiB | 10.44 MiB/s, done.


In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="danielliu1/liu_mesyngier_company_similarity_sae",
    filename="llama_features.pkl",
    repo_type="dataset"
)

llama_features.pkl:   0%|          | 0.00/29.4G [00:00<?, ?B/s]

# Feature Collection

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features")
df = ds["train"].to_pandas()[["__index_level_0__", "features"]]
df.to_pickle("/content/drive/MyDrive/company_similarity_sae/data/llama_features.pkl")

In [ ]:
from datasets import load_dataset
ds = load_dataset("marco-molinari/company_reports_with_features", streaming=True)
row = next(iter(ds["train"]))
print(row["__index_level_0__"])

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

0


# Main

This code only shows the output of each test, but doesn't save the pairs. This is necessary to include in the paper, to show that we iterated to through K.

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --top-k 500 750 1000 1250 1500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl

README.md: 100% 697/697 [00:00<00:00, 2.52MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:01<00:00, 76.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:02<00:00, 76.6MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:03<00:00, 41.4MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 110MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 87.7MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4942.77 examples/s]
README.md: 100% 592/592 [00:00<00:00, 346kB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 114MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:02<00:00, 116MB/s]
Generating train split: 100% 15002324/15002324 [00:07<00:00, 2096306.76 examples/s]
Scoring features (GPU): 100% 256/256 [01:44<00:00,  2.44it/s]

top-k = 500

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼────

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1750 2000 2250 2500 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl


top-k = 1750

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.1833       │ Top   0.5% = 0.3671  │ Top   0.5% = 0.4050  │
│       │              │ Top   1.0% = 0.3427  │ Top   1.0% = 0.3787  │
│       │              │ Top   2.0% = 0.3212  │ Top   2.0% = 0.3519  │
│       │              │ Top   5.0% = 0.2955  │ Top   5.0% = 0.3197  │
│       │              │ Top  10.0% = 0.2758  │ Top  10.0% = 0.2954  │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.1825       │ Top   0.5% = 0.3638  │ Top   0.5% = 0.4008  │
│       │              │ Top   1.0% = 0.3392  │ Top   1.0% = 0.3750  │
│       │              │ Top   2.0% = 0.3187  │ Top   2.0% = 0.3482  │
│       │              │ Top   5.0% = 0.2944  │ Top   5.0% = 0.3169  │
│       │              │ Top  10.0% = 0.2753  │ Top  10.0% = 0

This code saves the files, but specifically for k = 1250 (once we've established in the paper that k = 1250 has the best results.

In [ ]:
!python company_similarity_sae/new_approach/extract_features.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1250 \
    --norm-alpha 0 0.25 0.5 0.75 1.0 \
    --score-weight \
    --out-pairs /content/drive/MyDrive/company_similarity_sae/data/llama_pairs_sw.pkl \
    --out-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl


top-k = 1250

┌───────┬──────────────┬──────────────────────┬──────────────────────┐
│ alpha │ Spearman rho │ Precision-at-k (all) │ Precision-at-k (OOS) │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.0   │ 0.1826       │ Top   0.5% = 0.3646  │ Top   0.5% = 0.4025  │
│       │              │ Top   1.0% = 0.3411  │ Top   1.0% = 0.3762  │
│       │              │ Top   2.0% = 0.3191  │ Top   2.0% = 0.3495  │
│       │              │ Top   5.0% = 0.2927  │ Top   5.0% = 0.3165  │
│       │              │ Top  10.0% = 0.2733  │ Top  10.0% = 0.2909  │
├───────┼──────────────┼──────────────────────┼──────────────────────┤
│ 0.25  │ 0.1814       │ Top   0.5% = 0.3637  │ Top   0.5% = 0.4003  │
│       │              │ Top   1.0% = 0.3387  │ Top   1.0% = 0.3743  │
│       │              │ Top   2.0% = 0.3175  │ Top   2.0% = 0.3476  │
│       │              │ Top   5.0% = 0.2917  │ Top   5.0% = 0.3141  │
│       │              │ Top  10.0% = 0.2730  │ Top  10.0% = 0

## Changes:
1. We replaced PCA with a mean correlation-based feature selection.
2. We added a step that adjusts the features / activation vector for each company based on the score of each feature; that way, the important ones are emphasized.
3. We removed StandardScaler.
4. We changed the main evaluation method to be "precision-at-k", which measures the mean correlation of the highest-similarity pairs. We also did away with the mean correlation idea.
5. For the above ^ evaluation metric, we also used different k values (generally from 0.5% to 10%).
6. We also generally did away wtih using MST, since it's very vulnerable to tiny-cluster problem from the original paper. Instead, we use spectral clustering. Either way, this is not super relevant, since we're no longer relying on clustering.

# Final Results (Pairwise)

In [ ]:
!python company_similarity_sae/new_approach/evaluate.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1250 \
    --norm-alpha 0


Spearman rank correlation (all years)
┌──────────────────────────────┬────────────────┬───────────┐
│ Approach                     │   Spearman rho │   p-value │
├──────────────────────────────┼────────────────┼───────────┤
│ New approach (k=1250, α=0.0) │         0.1826 │         0 │
│ Parent paper (PCA 4000-dim)  │         0.0217 │         0 │
└──────────────────────────────┴────────────────┴───────────┘

Precision-at-k: mean return correlation, OOS 2014-2020
┌───────────┬────────────────┬────────────────┬────────────────┐
│ Cutoff    │   New approach │   Parent paper │ SIC baseline   │
├───────────┼────────────────┼────────────────┼────────────────┤
│ top 0.5%  │         0.4025 │         0.1592 │                │
│ top 1.0%  │         0.3762 │         0.1598 │ 0.2835         │
│ top 2.0%  │         0.3495 │         0.1755 │                │
│ top 5.0%  │         0.3165 │         0.1832 │                │
│ top 10.0% │         0.2909 │         0.1798 │                │
└───────────┴

# Final Results (Clustering)

In [ ]:
!python company_similarity_sae/new_approach/evaluate_clustering.py \
    --features-pkl {path} \
    --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
    --top-k 1250


Parent paper:
Building MSTs: 100% 25/25 [01:30<00:00,  3.62s/it]
Sweeping Theta: 100% 36/36 [00:47<00:00,  1.31s/it]

New method:
Building MSTs: 100% 25/25 [01:23<00:00,  3.35s/it]
Sweeping Theta: 100% 36/36 [00:17<00:00,  2.00it/s]

Unweighted MC(Gk)
┌─────────────────┬────────┬─────────────┐
│ Approach        │    OOS │   All years │
├─────────────────┼────────┼─────────────┤
│ New method      │ 0.3468 │      0.3287 │
│ Parent paper    │ 0.3789 │      0.3733 │
│ SIC code        │ 0.2429 │      0.2311 │
│ Population mean │ 0.1609 │      0.1609 │
└─────────────────┴────────┴─────────────┘

Weighted MC(Gk)
┌─────────────────┬────────┬─────────────┐
│ Approach        │    OOS │   All years │
├─────────────────┼────────┼─────────────┤
│ New method      │ 0.2255 │      0.232  │
│ Parent paper    │ 0.1612 │      0.1513 │
│ SIC code        │ 0.2771 │      0.252  │
│ Population mean │ 0.1609 │      0.1609 │
└─────────────────┴────────┴─────────────┘

New method OOS (θ=-1.90):
┌────────┬─────

# Ablation

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_pca_supervised.py \
--features-pkl {path} \
--sae-spearman-oos 0.2182 \
--sae-spearman-all 0.1826 \
--sae-lifts-oos "0.4025,0.3762,0.3495,0.3165,0.2909" \
--sae-lifts-all "0.3646,0.3411,0.3191,0.2927,0.2733"

README.md: 100% 697/697 [00:00<00:00, 3.90MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:02<00:00, 69.1MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:02<00:00, 76.6MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:01<00:00, 97.9MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 110MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 113MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4853.53 examples/s]
README.md: 100% 592/592 [00:00<00:00, 3.50MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:02<00:00, 159MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:01<00:00, 179MB/s]
Generating train split: 100% 15002324/15002324 [00:06<00:00, 2157693.32 examples/s]

Supervised PCA Results (OOS 2014-2020, 6,181,505 pairs):
┌────────────────┬────────────────┬───────────┬────────────────────┐
│ Method         │ Spearman rho   │ Cutoff    │ Mean correlation   │
├────────────────┼────────────────┼

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_unsupervised_sae.py \
--features-pkl {path} \
--sae-spearman-oos 0.2182 \
--sae-spearman-all 0.1826 \
--sae-lifts-oos "0.4025,0.3762,0.3495,0.3165,0.2909" \
--sae-lifts-all "0.3646,0.3411,0.3191,0.2927,0.2733" \
--parent-spearman-all 0.0217 \
--parent-lifts-oos "0.1592,0.1598,0.1755,0.1832,0.1798" \
--parent-spearman-oos 0.0278 \
--parent-lifts-all "0.1415,0.1445,0.1549,0.1665,0.1666"


Unsupervised SAE Results (OOS):
┌──────────────────┬────────────────┬───────────┬────────────────────┐
│ Method           │ Spearman rho   │ Cutoff    │ Mean correlation   │
├──────────────────┼────────────────┼───────────┼────────────────────┤
│ Unsupervised SAE │ 0.0502         │ top 0.5%  │ 0.4316             │
│ (cosine)         │                │ top 1.0%  │ 0.3801             │
│                  │                │ top 2.0%  │ 0.3285             │
│                  │                │ top 5.0%  │ 0.2675             │
│                  │                │ top 10.0% │ 0.2294             │
├──────────────────┼────────────────┼───────────┼────────────────────┤
│ Unsupervised SAE │ 0.0372         │ top 0.5%  │ 0.2005             │
│ (dot product)    │                │ top 1.0%  │ 0.2039             │
│                  │                │ top 2.0%  │ 0.2028             │
│                  │                │ top 5.0%  │ 0.1962             │
│                  │                │ top 10

In [ ]:
!python company_similarity_sae/new_approach/ablation/ablation_robustness.py \
--features-pkl {path} \
--load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl

┌───────────────────┬────────────────┬───────────┬────────────────────┐
│ Method            │ Spearman rho   │ Cutoff    │ Mean correlation   │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS baseline      │ 0.2182         │ top 0.5%  │ 0.4025             │
│                   │                │ top 1.0%  │ 0.3762             │
│                   │                │ top 2.0%  │ 0.3495             │
│                   │                │ top 5.0%  │ 0.3165             │
│                   │                │ top 10.0% │ 0.2909             │
├───────────────────┼────────────────┼───────────┼────────────────────┤
│ OOS excl. 2020    │ 0.1580         │ top 0.5%  │ 0.2530             │
│                   │                │ top 1.0%  │ 0.2425             │
│                   │                │ top 2.0%  │ 0.2316             │
│                   │                │ top 5.0%  │ 0.2177             │
│                   │                │ top 10.0% │ 0.2055       

# Plots

In [ ]:
!python company_similarity_sae/new_approach/norm_plot.py \
        --features-pkl {path} \
        --load-model /content/drive/MyDrive/company_similarity_sae/data/llama_selection_model.pkl \
        --top-k 1250 \
        --out-dir /content/drive/MyDrive/company_similarity_sae/figures

Loading features...
README.md: 100% 697/697 [00:00<00:00, 3.79MB/s]
data/train-00000-of-00005.parquet: 100% 140M/140M [00:01<00:00, 76.6MB/s]
data/train-00001-of-00005.parquet: 100% 154M/154M [00:01<00:00, 109MB/s]
data/train-00002-of-00005.parquet: 100% 158M/158M [00:01<00:00, 130MB/s]
data/train-00003-of-00005.parquet: 100% 155M/155M [00:01<00:00, 128MB/s]
data/train-00004-of-00005.parquet: 100% 159M/159M [00:01<00:00, 113MB/s]
Generating train split: 100% 27888/27888 [00:05<00:00, 4946.22 examples/s]
Selecting features...
  Selected 1250 features
Loading pairs...
README.md: 100% 592/592 [00:00<00:00, 3.82MB/s]
data/train-00000-of-00002.parquet: 100% 320M/320M [00:04<00:00, 72.6MB/s]
data/train-00001-of-00002.parquet: 100% 325M/325M [00:04<00:00, 70.5MB/s]
Generating train split: 100% 15002324/15002324 [00:06<00:00, 2163740.91 examples/s]
  14,920,749 pairs
Computing similarities...

  R² (norm product vs similarity): 0.3082
  Pearson r: 0.5552
  Spearman rho: 0.8057
  Regression: si

# Conclusions

Will do this